# Wildfire Training + Trained Eval (Hugging Face)

Use this notebook for GRPO training and trained-model evaluation on **Hugging Face GPU runtime**.

Cell order:
1. Run Cell 1 first each session (loads env vars + BASE).
2. Cells 2-7 are safe to re-run.

Tip: run the Kaggle untrained baseline notebook first so you have a clean before/after comparison.

In [ ]:
# Cell 1 (run first each session): env vars + constants (HF runtime)
import os

GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")
HF_TOKEN = os.environ.get("HF_TOKEN")

if not GITHUB_TOKEN:
    raise RuntimeError("Set GITHUB_TOKEN in the Hugging Face Space/Notebook secrets.")
if not HF_TOKEN:
    raise RuntimeError("Set HF_TOKEN in the Hugging Face Space/Notebook secrets.")

os.environ["HF_TOKEN"] = HF_TOKEN

REPO = "jhawaritvik/Scaler-hackathon"
REPO_URL = f"https://github.com/{REPO}.git"
BASE = "/home/user/app/wildfire-env"

print("Configured BASE:", BASE)
print("Tokens available: GITHUB_TOKEN + HF_TOKEN")

In [ ]:
# Cell 2: clone/update + install deps (safe to re-run)
import os
import glob
import subprocess
import sys

if not os.path.exists(BASE):
    subprocess.run(["git", "clone", f"https://{GITHUB_TOKEN}@github.com/{REPO}.git", BASE], check=True)

os.chdir(BASE)
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "checkout", "main"], check=True)
subprocess.run(["git", "pull", "--rebase", "origin", "main"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[train]"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "unsloth", "unsloth-zoo", "bitsandbytes", "xgrammar", "transformers", "trl"], check=True)

nvidia_lib_dirs = glob.glob("/usr/local/lib/python*/dist-packages/nvidia/*/lib")
if nvidia_lib_dirs:
    os.environ["LD_LIBRARY_PATH"] = ":".join(nvidia_lib_dirs) + ":" + os.environ.get("LD_LIBRARY_PATH", "")
    print("LD_LIBRARY_PATH updated with NVIDIA libs")

print("Install complete. cwd:", os.getcwd())

In [ ]:
# Cell 3: preflight for training stack
import subprocess
import sys

subprocess.run(["openenv", "validate", "."], check=True)
subprocess.run([sys.executable, "smoke_test.py"], check=True)
print("Preflight passed")

In [ ]:
# Cell 4: full GRPO training (resume-safe)
import subprocess
import sys

subprocess.run([sys.executable, "train_grpo.py"], check=True)
print("Training run finished")

In [ ]:
# Cell 5: evaluate trained policy (writes submission artifact)
import os
import subprocess
import sys

os.makedirs("submission_artifacts", exist_ok=True)
subprocess.run([
    sys.executable,
    "eval_policy.py",
    "--output",
    "submission_artifacts/eval_trained.json",
], check=True)

print("Wrote submission_artifacts/eval_trained.json")

In [ ]:
# Cell 6: generate submission plots and replay artifacts
import os
import subprocess
import sys

os.makedirs("replays", exist_ok=True)
subprocess.run([sys.executable, "plot_training_curves.py"], check=True)

print("Capturing trained replays...")
subprocess.run([sys.executable, "capture_replay.py", "--task", "easy", "--seed", "11", "--policy", "llm", "--adapter", "grpo_wildfire/best_adapter_easy", "--output", "replays/trained_easy_11.json"], check=True)
subprocess.run([sys.executable, "capture_replay.py", "--task", "medium", "--seed", "1", "--policy", "llm", "--adapter", "grpo_wildfire/best_adapter_medium", "--output", "replays/trained_medium_1.json"], check=True)
subprocess.run([sys.executable, "capture_replay.py", "--task", "hard", "--seed", "9", "--policy", "llm", "--adapter", "grpo_wildfire/best_adapter_hard", "--output", "replays/trained_hard_9.json"], check=True)

print("Capturing heuristic replays...")
subprocess.run([sys.executable, "capture_replay.py", "--task", "easy", "--seed", "11", "--policy", "heuristic", "--output", "replays/heuristic_easy_11.json"], check=True)
subprocess.run([sys.executable, "capture_replay.py", "--task", "medium", "--seed", "1", "--policy", "heuristic", "--output", "replays/heuristic_medium_1.json"], check=True)
subprocess.run([sys.executable, "capture_replay.py", "--task", "hard", "--seed", "9", "--policy", "heuristic", "--output", "replays/heuristic_hard_9.json"], check=True)
print("Replays and plots captured successfully.")


In [ ]:
# Cell 7: upload checkpoints to HF model repo (recommended)
import os
from huggingface_hub import HfApi, create_repo

HF_REPO_ID = os.environ.get("HF_REPO_ID", "Chunchunmaru-101/wildfire-grpo-checkpoints")
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("Set HF_TOKEN before upload.")

api = HfApi(token=HF_TOKEN)
create_repo(HF_REPO_ID, repo_type="model", exist_ok=True, token=HF_TOKEN)

api.upload_folder(
    folder_path="grpo_wildfire",
    repo_id=HF_REPO_ID,
    repo_type="model",
    commit_message="Upload wildfire GRPO checkpoints",
    ignore_patterns=["*.pyc", "__pycache__/*"],
)
print(f"Uploaded grpo_wildfire to https://huggingface.co/{HF_REPO_ID}")

In [ ]:
# Cell 8: optional push of trained artifacts to GitHub + quick view
import json
import subprocess

with open("submission_artifacts/eval_trained.json", "r", encoding="utf-8") as f:
    trained = json.load(f)

print("overall_mean:", trained.get("overall_mean"))
print(json.dumps(trained, indent=2)[:2000])

AUTO_PUSH = False  # set True only when ready to push from HF runtime
if AUTO_PUSH:
    subprocess.run(["git", "add", "submission_artifacts/", "replays/"], check=True)
    subprocess.run(["git", "commit", "-m", "Add trained eval artifacts and replays"], check=True)
    subprocess.run(["git", "push", "origin", "main"], check=True)
    print("Pushed eval_trained.json, replays, and plots to main")